In [138]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict,Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field


In [139]:
load_dotenv()

True

In [140]:
model= ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5,
)

In [141]:
class sentiementschema(BaseModel):
    sentiment : Literal['positive','negative']=Field(description='sentiment of the review')

In [142]:
structured_model=model.with_structured_output(sentiementschema)

In [143]:
prompt="the software is not worht it "
structured_model.invoke(prompt)

sentiementschema(sentiment='negative')

In [144]:
class reviewstate(TypedDict):
    review : str
    sentiment : Literal['positive','negative']
    diagnosis : dict
    response : str

In [145]:
def find_sentiment(state:reviewstate):
    prompt=f"for the follwing review ,find out  the sentiment {state['review']}"
    sentiment=structured_model.invoke(prompt).sentiment

    return {'sentiment':sentiment}

def check_sentiment(state:reviewstate) ->Literal['positive_response','run_diagnosis']:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state:reviewstate):
    prompt=f"write a thank you message for the given review {state['review']}"
    response=model.invoke(prompt).content
    
    return {'response':response}


def run_diagnosis(state:reviewstate):

    prompt=f"for the given review, tell the issue_type , tone and uregncy of the review{state['review']}"
    diagnosis=model.invoke(prompt).content
    return{'diagnosis':diagnosis}

def negative_response(state:reviewstate):

    prompt=f" write a relevant response as a support assistant to the negative review {state['review']} based on the diagnosis {state['diagnosis']}"
    response=model.invoke(prompt).content

    return {'response':response}

In [146]:
graph=StateGraph(reviewstate)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

In [147]:
graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('positive_response',END)
graph.add_edge('negative_response',END)

In [148]:
workflow=graph.compile()

In [149]:
initial_state={'review':'I have been using this app for a year now , at first it was working brilliantly but now its having so many glitches like home screen not loadingg , messages being not received , etc.'}
workflow.invoke(initial_state)

{'review': 'I have been using this app for a year now , at first it was working brilliantly but now its having so many glitches like home screen not loadingg , messages being not received , etc.',
 'sentiment': 'negative',
 'diagnosis': 'Based on the given review, here are the analysis results:\n\n1. **Issue Type:** The issue type is a **Technical Issue** or a **Performance Issue**. The user is experiencing various glitches and problems with the app, such as home screen not loading and messages not being received.\n\n2. **Tone:** The tone of the review is **Negative** and **Frustrated**. The user is expressing dissatisfaction with the app\'s performance and is frustrated that the issues have arisen after a year of using it.\n\n3. **Urgency:** The urgency of the review is **High**. The user is expressing a sense of urgency by stating that the app was working "brilliantly" in the past, implying that the current issues are a significant problem that needs to be addressed quickly.',
 'resp